# Módulo 2: Filtro Bayesiano Discreto — Práctica

**Módulo**: `2` | **Tipo**: PRÁCTICO | **Fecha**: 2026-05-20

In [1]:
import sys
import time
import numpy as np
import scipy
from scipy import stats
import plotly.graph_objects as go

np.random.seed(42)
assert sys.version_info >= (3, 12), f"Python 3.12+ requerido, tienes {sys.version}"
_t0 = time.time()
print(f"Python {sys.version_info.major}.{sys.version_info.minor}  |  NumPy {np.__version__}  |  SciPy {scipy.__version__}  ✓")

Python 3.14  |  NumPy 2.4.6  |  SciPy 1.17.1  ✓


In [2]:
# Parámetros del escenario de seguimiento en grilla 1D
N_POSICIONES = 20   # número de posiciones discretas
T_PASOS = 100       # número de pasos de tiempo
SIGMA_MOVIMIENTO = 1.0   # incertidumbre del movimiento (desv. estándar en unidades de celda)
SIGMA_SENSOR = 1.5       # ruido del sensor (desv. estándar)
VELOCIDAD = 1.0          # velocidad media del agente (posiciones por paso)

# Grilla de estados discretos
GRILLA = np.arange(N_POSICIONES, dtype=np.float64)

# Generación del estado verdadero (movimiento rectilíneo + ruido)
rng = np.random.default_rng(42)
estados_reales = np.zeros(T_PASOS, dtype=np.float64)
x = 2.0  # posición inicial
for k in range(T_PASOS):
    x = np.clip(x + VELOCIDAD + rng.normal(0, 0.3), 0, N_POSICIONES - 1)
    estados_reales[k] = x

# Observaciones ruidosas
observaciones = estados_reales + rng.normal(0, SIGMA_SENSOR, size=T_PASOS)
observaciones = np.clip(observaciones, 0, N_POSICIONES - 1)

print(f"Dataset: {T_PASOS} pasos en grilla de {N_POSICIONES} posiciones")
print(f"Estado inicial: {estados_reales[0]:.2f}  |  Estado final: {estados_reales[-1]:.2f}")
print(f"Observaciones: rango [{observaciones.min():.2f}, {observaciones.max():.2f}]")

Dataset: 100 pasos en grilla de 20 posiciones
Estado inicial: 3.09  |  Estado final: 19.00
Observaciones: rango [2.52, 19.00]


In [3]:
# [CELDA DE VALIDACIÓN — NO MODIFICAR]
# Referencia: filterpy.discrete_bayes
# Computa RMSE de referencia para comparar con la implementación del estudiante

try:
    from filterpy.discrete_bayes import predict as fp_predict, update as fp_update

    # Construir matriz de transición gaussiana
    def _build_T_matrix(n: int, sigma_mov: float) -> np.ndarray:
        T = np.zeros((n, n), dtype=np.float64)
        grilla_local = np.arange(n, dtype=np.float64)
        for j in range(n):
            kernel = stats.norm.pdf(grilla_local, loc=grilla_local[j] + VELOCIDAD, scale=sigma_mov)
            kernel_sum = kernel.sum()
            T[:, j] = kernel / kernel_sum if kernel_sum > 0 else np.ones(n) / n
        return T

    T_ref = _build_T_matrix(N_POSICIONES, SIGMA_MOVIMIENTO)
    b_ref = np.ones(N_POSICIONES, dtype=np.float64) / N_POSICIONES

    estimados_ref = np.zeros(T_PASOS, dtype=np.float64)
    for k in range(T_PASOS):
        b_ref = fp_predict(b_ref, offset=VELOCIDAD, kernel=stats.norm.pdf(
            np.arange(-N_POSICIONES+1, N_POSICIONES), scale=SIGMA_MOVIMIENTO
        ))
        likelihood_ref = stats.norm.pdf(GRILLA, loc=observaciones[k], scale=SIGMA_SENSOR)
        b_ref = fp_update(likelihood_ref, b_ref)
        estimados_ref[k] = float(GRILLA[np.argmax(b_ref)])

    rmse_ref = float(np.sqrt(np.mean((estimados_ref - estados_reales)**2)))
    print(f"=== REFERENCIA filterpy ===")
    print(f"RMSE MAP (filterpy): {rmse_ref:.4f}")
    _filterpy_ok = True

except ImportError:
    # filterpy no disponible — usar implementación interna como referencia
    def _build_T_matrix(n: int, sigma_mov: float) -> np.ndarray:
        T = np.zeros((n, n), dtype=np.float64)
        grilla_local = np.arange(n, dtype=np.float64)
        for j in range(n):
            kernel = stats.norm.pdf(grilla_local, loc=grilla_local[j] + VELOCIDAD, scale=sigma_mov)
            kernel_sum = kernel.sum()
            T[:, j] = kernel / kernel_sum if kernel_sum > 0 else np.ones(n) / n
        return T

    T_ref = _build_T_matrix(N_POSICIONES, SIGMA_MOVIMIENTO)
    b_ref_int = np.ones(N_POSICIONES, dtype=np.float64) / N_POSICIONES
    estimados_ref_int = np.zeros(T_PASOS, dtype=np.float64)
    for k in range(T_PASOS):
        b_ref_int = T_ref @ b_ref_int
        likelihood_int = stats.norm.pdf(GRILLA, loc=observaciones[k], scale=SIGMA_SENSOR)
        b_ref_int = likelihood_int * b_ref_int
        b_ref_int = b_ref_int / b_ref_int.sum()
        estimados_ref_int[k] = float(GRILLA[np.argmax(b_ref_int)])

    rmse_ref = float(np.sqrt(np.mean((estimados_ref_int - estados_reales)**2)))
    estimados_ref = estimados_ref_int
    print(f"=== REFERENCIA interna (filterpy no disponible) ===")
    print(f"RMSE MAP (ref. interna): {rmse_ref:.4f}")
    _filterpy_ok = False

=== REFERENCIA interna (filterpy no disponible) ===
RMSE MAP (ref. interna): 0.3233


## Implementación del Filtro Bayesiano Discreto

**Objetivo**: Implementar el ciclo predict–update para seguimiento en la grilla 1D.

**Algoritmo**:
1. Construye la matriz de transición $T$ (modelo Gaussiano de movimiento)
2. Inicializa la creencia $b$ como distribución uniforme
3. Para cada paso: predicción → corrección → registrar estimado MAP

**Recuerda**:
- Predicción: `b_bar = T @ b`
- Corrección: `b = likelihood * b_bar; b = b / b.sum()`
- Estimado MAP: `grilla[np.argmax(b)]`

In [4]:
# TODO: implementar el Filtro Bayesiano Discreto
# Parte 1 — Construir la matriz de transición T (N x N)
# T[i, j] = P(x'=grilla[i] | x=grilla[j]) — modelo Gaussiano centrado en grilla[j]+VELOCIDAD

def build_T_matrix(n: int, grilla: np.ndarray, vel: float, sigma_mov: float) -> np.ndarray:
    # TODO: llenar la matriz de transición
    # Pista: para cada columna j, el kernel es stats.norm.pdf(grilla, loc=grilla[j]+vel, scale=sigma_mov)
    T = np.zeros((n, n), dtype=np.float64)
    for j in range(n):
        kernel = stats.norm.pdf(grilla, loc=grilla[j] + vel, scale=sigma_mov)
        T[:, j] = kernel / kernel.sum()   # ← TODO: normalizar cada columna
    return T

T_student = build_T_matrix(N_POSICIONES, GRILLA, VELOCIDAD, SIGMA_MOVIMIENTO)

# Verificar que columnas suman 1
assert np.allclose(T_student.sum(axis=0), 1.0, atol=1e-6), "Columnas de T no suman 1"
print(f"T_student shape: {T_student.shape}  ✓")

# Parte 2 — Ciclo predict–update
b = np.ones(N_POSICIONES, dtype=np.float64) / N_POSICIONES   # prior uniforme
estimados_student = np.zeros(T_PASOS, dtype=np.float64)
history_belief = np.zeros((T_PASOS, N_POSICIONES), dtype=np.float64)

for k in range(T_PASOS):
    # TODO: paso de predicción
    b_bar = T_student @ b           # Chapman-Kolmogorov: b_bar = T @ b

    # TODO: calcular likelihood para la observación observaciones[k]
    likelihood = stats.norm.pdf(GRILLA, loc=observaciones[k], scale=SIGMA_SENSOR)

    # TODO: paso de corrección y normalización
    b = likelihood * b_bar          # regla de Bayes (sin normalizar)
    b = b / b.sum()                 # normalizar

    # Registrar estimado MAP
    estimados_student[k] = GRILLA[np.argmax(b)]
    history_belief[k] = b

rmse_student = float(np.sqrt(np.mean((estimados_student - estados_reales)**2)))
print(f"RMSE MAP (tu implementación): {rmse_student:.4f}")

T_student shape: (20, 20)  ✓
RMSE MAP (tu implementación): 0.3233


In [5]:
# Validación numérica — descomentar cuando hayas completado los TODO
# TOL = 1e-6
# assert abs(rmse_student - rmse_ref) < TOL, (
#     f"RMSE_student={rmse_student:.6f} difiere de RMSE_ref={rmse_ref:.6f} "
#     f"(diferencia={abs(rmse_student - rmse_ref):.2e}, tolerancia={TOL})"
# )
# assert rmse_student < 2.0, f"RMSE {rmse_student:.4f} > 2.0 — el filtro no converge"
print("→ Descomenta el assert cuando hayas implementado los TODO.")
print(f"  Referencia:       RMSE = {rmse_ref:.4f}")
print(f"  Tu implementación: RMSE = {rmse_student:.4f}")

→ Descomenta el assert cuando hayas implementado los TODO.
  Referencia:       RMSE = 0.3233
  Tu implementación: RMSE = 0.3233


In [6]:
# Visualización Plotly — mapa de calor belief(t, x) mostrando convergencia
fig = go.Figure()

# Mapa de calor de la creencia a lo largo del tiempo
fig.add_trace(go.Heatmap(
    z=history_belief.T,        # shape (N_POSICIONES, T_PASOS)
    x=np.arange(T_PASOS),
    y=GRILLA,
    colorscale="Blues",
    colorbar=dict(title="P(x|z₁:ₖ)", tickformat=".2f"),
    name="Belief",
))

# Estado real
fig.add_trace(go.Scatter(
    x=np.arange(T_PASOS),
    y=estados_reales,
    mode="lines",
    name="Estado real",
    line=dict(color="red", width=2),
))

# Estimado MAP
fig.add_trace(go.Scatter(
    x=np.arange(T_PASOS),
    y=estimados_student,
    mode="lines",
    name=f"MAP (RMSE={rmse_student:.2f})",
    line=dict(color="lime", width=1.5, dash="dot"),
))

fig.update_layout(
    title="Módulo 2 — Filtro Bayesiano Discreto: Belief(t, x)",
    xaxis_title="Tiempo [pasos]",
    yaxis_title="Posición [celdas]",
    legend_title="Trayectoria",
    template="plotly_white",
    width=800, height=450,
)
fig.show()

## Ejercicios guiados (base, ≤ 30 min)

**Ejercicio 1 — Convergencia MAP**: ¿En cuántos pasos el error MAP baja por primera vez por debajo de 2 celdas? Busca el primer índice donde `abs(estimados_student[k] - estados_reales[k]) < 2.0`.

**Ejercicio 2 — Efecto del sensor**: Prueba con `SIGMA_SENSOR = 0.5` y luego con `SIGMA_SENSOR = 3.0`. ¿Cómo cambia el RMSE final? ¿Por qué un sensor más preciso ayuda más al principio que al final?

**Ejercicio 3 — Entropía del belief**: Calcula la entropía $H[b_k] = -\sum_x b_k[x] \log b_k[x]$ para cada paso. Grafica la entropía vs. tiempo. ¿Qué observas en los primeros 10 pasos?

In [7]:
# Ejercicio 3 — Entropía del belief
entropias = np.zeros(T_PASOS, dtype=np.float64)
for k in range(T_PASOS):
    b_k = history_belief[k]
    # Evitar log(0): filtrar valores muy pequeños
    mask = b_k > 1e-300
    entropias[k] = -float(np.sum(b_k[mask] * np.log(b_k[mask])))

print(f"Entropía inicial (uniforme): {entropias[0]:.4f}  (teórico = log({N_POSICIONES}) = {np.log(N_POSICIONES):.4f})")
print(f"Entropía final:              {entropias[-1]:.4f}")
print(f"Paso en que H baja de log(N/2)={np.log(N_POSICIONES/2):.2f}: {np.argmax(entropias < np.log(N_POSICIONES/2))}")

Entropía inicial (uniforme): 1.6789  (teórico = log(20) = 2.9957)
Entropía final:              0.5855
Paso en que H baja de log(N/2)=2.30: 0


## Reto (opcional — sin andamiaje)

Implementa el mismo filtro pero con un **modelo de movimiento bimodal**: el agente puede moverse +1 con probabilidad 0.6 o +2 con probabilidad 0.4. Esto hace que el belief pueda ser multimodal (dos "picos" en la distribución).

- Construye la nueva T_bimodal
- Ejecuta el filtro con `SIGMA_SENSOR = 1.0` y `T=50` pasos
- Grafica el heatmap e identifica los pasos donde el belief tiene dos máximos locales

In [8]:
# Smoke-test: verificar tiempo de ejecución total
_tiempo = time.time() - _t0
assert _tiempo < 60.0, f"Notebook tardó {_tiempo:.1f}s (máx 60s)"
print(f"Tiempo total de ejecución: {_tiempo:.2f}s  ✓")

Tiempo total de ejecución: 2.99s  ✓


## ¿Qué aprendiste?

Programando este notebook aprendimos que:

1. El filtro bayesiano discreto converge rápidamente desde un prior uniforme si el sensor es suficientemente preciso — en pocos pasos el belief se concentra alrededor del estado real.
2. El mapa de calor `belief(t, x)` es una herramienta poderosa para visualizar la evolución de la incertidumbre — muestra exactamente cuándo y cómo el filtro "aprende".
3. El RMSE final depende principalmente del balance entre `SIGMA_MOVIMIENTO` y `SIGMA_SENSOR`: un sensor preciso compensa un modelo de movimiento incierto.

## ¿Qué sigue?

En el **Módulo 3a: Filtro de Kalman Lineal** extenderemos el mismo ciclo predict–update a espacios de estados **continuos y Gaussianos**. En lugar de propagar un vector de N probabilidades, propagaremos solo la media $\hat{x}$ y la covarianza $P$ de una distribución Gaussiana — reduciendo el costo computacional de $O(N^2)$ a $O(n^3)$ donde $n$ es la dimensión del estado.